In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fastapi uvicorn websockets pydub soundfile pyngrok nest-asyncio
!apt-get install -y ffmpeg
!pip install pydub soundfile pyngrok uvicorn fastapi nest-asyncio websockets
!pip install munch
!pip install -q parallel_wavegan munch pyngrok uvicorn fastapi pydub soundfile nest-asyncio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.2 MB/s eta 0:00:00


In [3]:
%%writefile server.py
import sys, os, io, yaml, torch, librosa
import numpy as np
import soundfile as sf
import torchaudio
from pydub import AudioSegment
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware

WORK_DIR = '/content/drive/MyDrive/Github_G8/Accent-Conversion'
sys.path.append(WORK_DIR)
from models import build_model
from Utils.JDC.model import JDCNet
from parallel_wavegan.utils import load_model

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("正在加载模型...")
with open(os.path.join(WORK_DIR, 'Configs/config.yml')) as f:
    config = yaml.safe_load(f)

generator, _, style_encoder = build_model(model_params=config["model_params"])
state_dict = torch.load(os.path.join(WORK_DIR, 'Models/epoch_00150.pth'), map_location=device)
generator.load_state_dict(state_dict['generator'])
style_encoder.load_state_dict(state_dict['style_encoder'])
generator.to(device).eval()
style_encoder.to(device).eval()

F0_model = JDCNet(num_class=1, num_class_q=256)
F0_model.load_state_dict(torch.load(os.path.join(WORK_DIR, 'Utils/JDC/bst.t7'), map_location=device)['net'])
F0_model.to(device).eval()

vocoder = load_model(os.path.join(WORK_DIR, "Vocoder/checkpoint-400000steps.pkl")).to(device).eval()
vocoder.remove_weight_norm()

to_mel = torchaudio.transforms.MelSpectrogram(n_mels=80, n_fft=2048, win_length=1200, hop_length=300).to(device)
def compute_mel(wav_tensor):
    mel_tensor = to_mel(wav_tensor)
    return (torch.log(1e-5 + mel_tensor) - (-4)) / 4

# 【注意】：确保这个路径是你想要变成的那个人的声音文件！
TARGET_AUDIO_PATH = os.path.join(WORK_DIR, "Data/p225/1.wav")
with torch.no_grad():
    ref_wav, _ = librosa.load(TARGET_AUDIO_PATH, sr=24000)
    ref_wav = librosa.effects.trim(ref_wav, top_db=30)[0]
    ref_mel = compute_mel(torch.tensor(ref_wav, device=device).unsqueeze(0))
    TARGET_STYLE = style_encoder(ref_mel.unsqueeze(0))

print("✅ AI 灵魂注入完毕！等待连接...")

@app.websocket("/ws/vc")
async def websocket_vc(websocket: WebSocket):
    await websocket.accept()
    print("\n🎉 前端已连接，开始变声！")

    while True:
        try: # ！！！核心修改：把 try 放在 while 里面 ！！！
            webm_bytes = await websocket.receive_bytes()
            audio_segment = AudioSegment.from_file(io.BytesIO(webm_bytes), format="webm")
            audio_segment = audio_segment.set_channels(1).set_frame_rate(24000)
            samples = np.array(audio_segment.get_array_of_samples(), dtype=np.float32) / 32768.0

            # ！！！核心修改：如果切片短于 0.5 秒 (12000 个采样点)，直接扔掉，防止模型报错崩溃！
            if len(samples) < 12000:
                continue

            with torch.no_grad():
                source_wav = torch.tensor(samples, dtype=torch.float32, device=device).unsqueeze(0)
                mel_tensor = compute_mel(source_wav)
                f0, _ = F0_model.compute_real_f0(source_wav)

                # 对齐时间维度 (防止卷积报错)
                min_len = min(mel_tensor.size(-1), f0.size(-1))
                mel_tensor = mel_tensor[..., :min_len]
                f0 = f0[..., :min_len]

                converted_mel = generator(mel_tensor.unsqueeze(0), f0.unsqueeze(0), TARGET_STYLE)
                c = converted_mel.squeeze(0).transpose(0, 1)
                converted_samples = vocoder.inference(c).squeeze().cpu().numpy()

            out_io = io.BytesIO()
            sf.write(out_io, converted_samples, 24000, format='WAV')
            await websocket.send_bytes(out_io.getvalue())

        except WebSocketDisconnect:
            print("网页端断开了连接")
            break # 只有真正断开才退出循环
        except Exception as e:
            print(f"⚠️ 遇到瑕疵音频块，已跳过，报错: {e}")
            # 不 break，继续接听下一秒！

Writing server.py


In [4]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio

nest_asyncio.apply()

# 0. 【极其重要】先杀掉之前残留的 ngrok 进程，防止“隧道已存在”报错！
ngrok.kill()

# 1. 填入你的专属 Token
ngrok.set_auth_token("3BUOT4U5ZAcooHpRtdJlypPVZmo_5YiZj2TkVKS8A2WpkcNKV")

# 2. 建立隧道，并强制绑定你的固定域名！
try:
    tunnel = ngrok.connect(8000, domain="indistinctively-nonhesitant-briggs.ngrok-free.dev")
    print("==================================================")
    print(f"✅ 隧道打通成功！")
    print(f"👉 请确保你 HTML 里的地址是: wss://indistinctively-nonhesitant-briggs.ngrok-free.dev/ws/vc")
    print("==================================================")
except Exception as e:
    print(f"隧道建立失败，报错信息: {e}")
    print("请检查你的 Token 是否正确，或者该域名是否已经被其他设备占用。")

# 3. 启动 FastAPI 服务器！(会一直转圈并在下方打印日志)
# 注意：如果这里报错“端口 8000 被占用”，请点击 Colab 顶部的“代码执行程序” -> “重启会话”，然后再重新运行！
config = uvicorn.Config("server:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

✅ 隧道打通成功！
👉 请确保你 HTML 里的地址是: wss://indistinctively-nonhesitant-briggs.ngrok-free.dev/ws/vc


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


正在加载模型...


TypeError: build_model() got an unexpected keyword argument 'model_params'

In [5]:
import sys, os, io, yaml, torch, librosa, asyncio, nest_asyncio
import numpy as np
import soundfile as sf
import torchaudio
from pydub import AudioSegment
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# ==========================================
# 1. 尝试导入模型 (如果有错会直接在这里爆红)
# ==========================================
WORK_DIR = '/content/drive/MyDrive/Github_G8/Accent-Conversion'
sys.path.append(WORK_DIR)

print("🔍 阶段 1: 检查环境与模型库...")
try:
    from models import build_model
    from Utils.JDC.model import JDCNet
    from parallel_wavegan.utils import load_model
    print("✅ 模型库导入成功！")
except Exception as e:
    print(f"❌ 导入失败，请检查路径是否正确: {e}")

# ==========================================
# 2. 尝试加载权重
# ==========================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔍 阶段 2: 正在使用 {device} 加载几十个G的模型权重，请耐心等待...")

try:
    with open(os.path.join(WORK_DIR, 'Configs/config.yml')) as f:
        config = yaml.safe_load(f)

    generator, _, style_encoder = build_model(model_params=config["model_params"])
    state_dict = torch.load(os.path.join(WORK_DIR, 'Models/epoch_00150.pth'), map_location=device)
    generator.load_state_dict(state_dict['generator'])
    style_encoder.load_state_dict(state_dict['style_encoder'])
    generator.to(device).eval()
    style_encoder.to(device).eval()

    F0_model = JDCNet(num_class=1, num_class_q=256)
    F0_model.load_state_dict(torch.load(os.path.join(WORK_DIR, 'Utils/JDC/bst.t7'), map_location=device)['net'])
    F0_model.to(device).eval()

    vocoder = load_model(os.path.join(WORK_DIR, "Vocoder/checkpoint-400000steps.pkl")).to(device).eval()
    vocoder.remove_weight_norm()

    to_mel = torchaudio.transforms.MelSpectrogram(n_mels=80, n_fft=2048, win_length=1200, hop_length=300).to(device)
    def compute_mel(wav_tensor):
        mel_tensor = to_mel(wav_tensor)
        return (torch.log(1e-5 + mel_tensor) - (-4)) / 4

    # ⚠️⚠️⚠️ 必须修改这里：你要变成的声音文件路径！
    TARGET_AUDIO_PATH = os.path.join(WORK_DIR, "Data/p225/p225_001.wav")
    print(f"提取目标音色: {TARGET_AUDIO_PATH}")

    with torch.no_grad():
        ref_wav, _ = librosa.load(TARGET_AUDIO_PATH, sr=24000)
        ref_wav = librosa.effects.trim(ref_wav, top_db=30)[0]
        ref_mel = compute_mel(torch.tensor(ref_wav, device=device).unsqueeze(0))
        TARGET_STYLE = style_encoder(ref_mel.unsqueeze(0))

    print("✅✅✅ 阶段 2 完成！灵魂注入成功！")
except Exception as e:
    print(f"❌❌❌ 模型加载彻底失败，报错原因: {e}")

# ==========================================
# 3. 启动 FastAPI 和 Ngrok 隧道
# ==========================================
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.websocket("/ws/vc")
async def websocket_vc(websocket: WebSocket):
    await websocket.accept()
    print("\n🟢 前端已连接！你可以点击录音了。")

    while True:
        try:
            webm_bytes = await websocket.receive_bytes()
            print("▶️ 收到前端音频，开始解码...")

            audio_segment = AudioSegment.from_file(io.BytesIO(webm_bytes), format="webm")
            audio_segment = audio_segment.set_channels(1).set_frame_rate(24000)
            samples = np.array(audio_segment.get_array_of_samples(), dtype=np.float32) / 32768.0
            print(f"   音频长度: {len(samples)} 个采样点")

            if len(samples) < 12000:
                print("   ⚠️ 声音太短，忽略本次请求。")
                continue

            print("⚙️ 正在送入 StarGANv2-VC 进行变声...")
            with torch.no_grad():
                source_wav = torch.tensor(samples, dtype=torch.float32, device=device).unsqueeze(0)
                mel_tensor = compute_mel(source_wav)
                f0, _ = F0_model.compute_real_f0(source_wav)

                min_len = min(mel_tensor.size(-1), f0.size(-1))
                mel_tensor = mel_tensor[..., :min_len]
                f0 = f0[..., :min_len]

                converted_mel = generator(mel_tensor.unsqueeze(0), f0.unsqueeze(0), TARGET_STYLE)
                c = converted_mel.squeeze(0).transpose(0, 1)
                converted_samples = vocoder.inference(c).squeeze().cpu().numpy()

            print("✅ 变声完成！正在发回网页...")
            out_io = io.BytesIO()
            sf.write(out_io, converted_samples, 24000, format='WAV')
            await websocket.send_bytes(out_io.getvalue())

        except WebSocketDisconnect:
            print("🔴 网页端断开了连接")
            break
        except Exception as e:
            print(f"❌ 变声过程中发生致命错误: {e}")

# 清理旧隧道并重新绑定
print("\n🔍 阶段 3: 正在启动网络隧道...")
ngrok.kill()
ngrok.set_auth_token("3BUOT4U5ZAcooHpRtdJlypPVZmo_5YiZj2TkVKS8A2WpkcNKV")
try:
    tunnel = ngrok.connect(8000, domain="indistinctively-nonhesitant-briggs.ngrok-free.dev")
    print("✅ 隧道就绪！")
except Exception as e:
    print(f"⚠️ 隧道警告: {e}")

print("\n🚀🚀🚀 服务器已全面启动，请去网页操作！")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning") # 把app直接传进去，告别缓存！
server = uvicorn.Server(config)
await server.serve()

🔍 阶段 1: 检查环境与模型库...
✅ 模型库导入成功！
🔍 阶段 2: 正在使用 cuda 加载几十个G的模型权重，请耐心等待...
❌❌❌ 模型加载彻底失败，报错原因: build_model() got an unexpected keyword argument 'model_params'

🔍 阶段 3: 正在启动网络隧道...
✅ 隧道就绪！

🚀🚀🚀 服务器已全面启动，请去网页操作！

🟢 前端已连接！你可以点击录音了。
▶️ 收到前端音频，开始解码...
   音频长度: 67680 个采样点
⚙️ 正在送入 StarGANv2-VC 进行变声...
❌ 变声过程中发生致命错误: name 'compute_mel' is not defined
🔴 网页端断开了连接


In [6]:
%cd /content/drive/MyDrive/Github_G8/Accent-Conversion//

/content/drive/MyDrive/Github_G8/Accent-Conversion


In [ ]:
!python train.py